<a href="https://colab.research.google.com/github/TerteryanTatev/Decision-Support-and-Expert-Systems/blob/main/Expert-Concordance-Analysis/expert_concordance_kendall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def input_rankings():
    K = int(input("Գործոնների քանակ K = "))
    N = int(input("Փորձագետների քանակ N = "))

    rankings = np.zeros((K, N))

    print("\nՄուտքագրիր դասակարգումները 1..K (առանց կրկնության)")

    for j in range(N):
        while True:
            try:
                values = list(map(int, input(f"Փորձագետ {j+1}: ").split()))

                if len(values) != K:
                    raise ValueError("Սխալ քանակ")

                if sorted(values) != list(range(1, K + 1)):
                    raise ValueError("Պետք է լինի 1..K")

                rankings[:, j] = values
                break

            except ValueError as e:
                print("Սխալ:", e)

    return rankings

In [ ]:
def rankings_to_pairwise(rankings):
    K, N = rankings.shape
    pairwise = np.zeros((N, K, K))

    for e in range(N):
        for i in range(K):
            for j in range(K):
                if i == j:
                    continue

                pairwise[e, i, j] = 1 if rankings[i, e] < rankings[j, e] else 0

    return pairwise

In [ ]:
def pairwise_to_scores(pairwise):
    return np.sum(pairwise, axis=2)

In [ ]:
def scores_to_rankings(scores):
    N, K = scores.shape
    rankings = np.zeros((K, N))

    for e in range(N):
        rankings[:, e] = np.argsort(np.argsort(-scores[e])) + 1

    return rankings

In [ ]:
def compute_final_ranking(rankings):
    R = np.sum(rankings, axis=1)
    final_ranks = np.argsort(np.argsort(R)) + 1
    return R, final_ranks

In [ ]:
def expert_matches(rankings, final_ranks):
    K, N = rankings.shape
    matches = []

    for j in range(N):
        t_j = np.sum(rankings[:, j] == final_ranks)
        matches.append(t_j)
        print(f"Փորձագետ {j+1}: {t_j}/{K}")

    return matches

In [ ]:
def compute_Tj(matches):
    T_sum = 0

    for i, t in enumerate(matches):
        Tj = t**3 - t
        T_sum += Tj
        print(f"T{i+1} = {Tj}")

    return T_sum

In [ ]:
def kendall_W(rankings, R, T_sum):
    K, N = rankings.shape

    R_bar = 0.5 * N * (K + 1)
    S = np.sum((R - R_bar) ** 2)

    W = 12 * S / (N**2 * (K**3 - K) - N * T_sum)

    return W, S

In [ ]:
def build_table(rankings, R, final_ranks):
    K, N = rankings.shape
    data = {}

    for j in range(N):
        data[f"Փորձագետ {j+1}"] = rankings[:, j]

    data["R"] = R
    data["Final"] = final_ranks

    df = pd.DataFrame(data)
    df.index = [f"Գործոն {i+1}" for i in range(K)]
    df = df.sort_values("Final")

    return df

In [ ]:
def build_average_pairwise(pairwise):
    avg = np.mean(pairwise, axis=0)

    df = pd.DataFrame(avg)
    K = avg.shape[0]

    df.index = [f"F{i+1}" for i in range(K)]
    df.columns = [f"F{j+1}" for j in range(K)]

    print("\n Միջին Pairwise matrix (բոլոր փորձագետներ)")
    print(df)

    return df

In [ ]:
def build_pairwise_list_tables(pairwise):
    N, K, _ = pairwise.shape
    results = []

    for e in range(N):
        rows = []

        for i in range(K):
            for j in range(i+1, K):
                rows.append({
                    "i": f"F{i+1}",
                    "j": f"F{j+1}",
                    "i > j (1=yes)": int(pairwise[e, i, j])
                })

        df = pd.DataFrame(rows)
        print(f"\n Փորձագետ {e+1} Pairwise (list ձև)")
        print(df)

        results.append(df)

    return results

In [ ]:
rankings_input = input_rankings()

pairwise = rankings_to_pairwise(rankings_input)

scores = pairwise_to_scores(pairwise)

rankings = scores_to_rankings(scores)

R, final_ranks = compute_final_ranking(rankings)

print("\nՎերջնական դասակարգում")
for i, r in enumerate(final_ranks):
    print(f"Գործոն {i+1} → {r}")

matches = expert_matches(rankings, final_ranks)

T_sum = compute_Tj(matches)

W, S = kendall_W(rankings, R, T_sum)

print("\nS =", round(S, 3))
print("ΣTj =", T_sum)
print("Kendall W =", round(W, 4))

build_pairwise_list_tables(pairwise)
build_average_pairwise(pairwise)
df = build_table(rankings, R, final_ranks)
print("\nԱղյուսակ:")
print(df)


Գործոնների քանակ K = 5
Փորձագետների քանակ N = 5

Մուտքագրիր դասակարգումները 1..K (առանց կրկնության)
Փորձագետ 1: 1 2 3 4 5
Փորձագետ 2: 2 1 4 3 5
Փորձագետ 3: 2 3 1 4 5
Փորձագետ 4: 1 3 4 2 5
Փորձագետ 5: 1 3 2 5 4

Վերջնական դասակարգում
Գործոն 1 → 1
Գործոն 2 → 2
Գործոն 3 → 3
Գործոն 4 → 4
Գործոն 5 → 5
Փորձագետ 1: 5/5
Փորձագետ 2: 1/5
Փորձագետ 3: 2/5
Փորձագետ 4: 2/5
Փորձագետ 5: 1/5
T1 = 120
T2 = 0
T3 = 6
T4 = 6
T5 = 0

S = 164.0
ΣTj = 132
Kendall W = 0.841

 Փորձագետ 1 Pairwise (list ձև)
    i   j  i > j (1=yes)
0  F1  F2              1
1  F1  F3              1
2  F1  F4              1
3  F1  F5              1
4  F2  F3              1
5  F2  F4              1
6  F2  F5              1
7  F3  F4              1
8  F3  F5              1
9  F4  F5              1

 Փորձագետ 2 Pairwise (list ձև)
    i   j  i > j (1=yes)
0  F1  F2              0
1  F1  F3              1
2  F1  F4              1
3  F1  F5              1
4  F2  F3              1
5  F2  F4              1
6  F2  F5              1
7  F3  

-----------------------------------------------------------------------------------

In [ ]:

def input_rankings():
    K = int(input("Գործոնների քանակ K = "))
    N = int(input("Փորձագետների քանակ N = "))

    rankings = np.zeros((K, N))

    print("\nՅուրաքանչյուր փորձագետ մուտքագրում է դասակարգում 1..K առանց կրկնության")

    for j in range(N):
        while True:
            try:
                values = list(map(int, input(f"Փորձագետ {j+1}: ").split()))

                if len(values) != K:
                    raise ValueError("Պետք է մուտքագրել ճիշտ քանակ թվեր")

                if sorted(values) != list(range(1, K+1)):
                    raise ValueError("Պետք է լինեն 1..K առանց կրկնության")

                rankings[:, j] = values
                break

            except ValueError as e:
                print("Սխալ:", e)

    return rankings


In [ ]:

def compute_final_ranking(rankings):
    R = np.sum(rankings, axis=1)
    final_ranks = np.argsort(np.argsort(R)) + 1
    return R, final_ranks

In [ ]:

def expert_matches(rankings, final_ranks):
    K, N = rankings.shape
    matches = []

    print("\nՀամընկնումներ վերջնական դասակարգման հետ")

    for j in range(N):
        t_j = np.sum(rankings[:, j] == final_ranks)
        matches.append(t_j)
        print(f"Փորձագետ {j+1}: {t_j} / {K}")

    return matches

In [ ]:

def compute_Tj(matches):
    T_sum = 0
    print("\nTj հաշվարկ")

    for i, t in enumerate(matches):
        Tj = t**3 - t
        T_sum += Tj
        print(f"T{i+1} = {t}^3 - {t} = {Tj}")

    print("ΣTj =", T_sum)
    return T_sum

In [ ]:

def kendall_W(rankings, R, T_sum):
    K, N = rankings.shape

    R_bar = 0.5 * N * (K+1)
    S = np.sum((R - R_bar)**2)
    W = 12 * S / (N**2 * (K**3 - K) - N * T_sum)
    print('a=',R_bar)
    return W, S

In [ ]:

rankings = input_rankings()
K, N = rankings.shape

R, final_ranks = compute_final_ranking(rankings)

print("\nRi (տողերի գումարներ)")
for i, r in enumerate(R):
    print(f"Գործոն {i+1}: {r}")

print("\nՎերջնական դասակարգում")
for i, r in enumerate(final_ranks):
    print(f"Գործոն {i+1} → տեղ {r}")

m = expert_matches(rankings, final_ranks)

T_sum = compute_Tj(m)

W, S = kendall_W(rankings, R, T_sum)

print("S  =", round(S,3))
print("ΣTj =", T_sum)

print("\nKendall W =", round(W,4))


Գործոնների քանակ K = 6
Փորձագետների քանակ N = 5

Յուրաքանչյուր փորձագետ մուտքագրում է դասակարգում 1..K առանց կրկնության
Փորձագետ 1: 1 2 3 4 5 6
Փորձագետ 2: 6 5 4 3 2 1
Փորձագետ 3: 1 2 3 4 6 5


KeyboardInterrupt: Interrupted by user

In [ ]:
if W>0.8:
  print('lav')
elif W>0.7:
  print('mij')
else:
  print('vat')


vat


-----------------------------------------------------------------------------------------------------


In [ ]:


def input_pairwise():
    K = int(input("Գործոնների քանակ K = "))
    N = int(input("Փորձագետների քանակ N = "))

    pairwise = np.zeros((N, K, K))

    print("\nՄուտքագրիր միայն i < j զույգերը")
    print("Եթե i-ն գերադասելի է j-ից → 1, հակառակ դեպքում 0\n")

    for e in range(N):
        print(f"\nՓորձագետ {e+1}")
        for i in range(K):
            for j in range(i+1, K):
                val = int(input(f"{i+1} vs {j+1}: "))

                pairwise[e, i, j] = val
                pairwise[e, j, i] = 1 - val

    return pairwise


def pairwise_to_scores(pairwise):
    scores = np.sum(pairwise, axis=2)
    return scores


def scores_to_rankings(scores):
    N, K = scores.shape
    rankings = np.zeros((K, N))

    for e in range(N):
        rankings[:, e] = np.argsort(np.argsort(-scores[e])) + 1

    return rankings


def compute_final_ranking(rankings):
    R = np.sum(rankings, axis=1)
    final_ranks = np.argsort(np.argsort(R)) + 1
    return R, final_ranks


def expert_matches(rankings, final_ranks):
    K, N = rankings.shape
    matches = []

    for j in range(N):
        t_j = np.sum(rankings[:, j] == final_ranks)
        matches.append(t_j)
        print(f"Փորձագետ {j+1}: {t_j}/{K}")

    return matches


def compute_Tj(matches):
    T_sum = 0
    for i, t in enumerate(matches):
        Tj = t**3 - t
        T_sum += Tj
        print(f"T{i+1} = {Tj}")
    return T_sum


def kendall_W(rankings, R, T_sum):
    K, N = rankings.shape

    R_bar = 0.5 * N * (K + 1)
    S = np.sum((R - R_bar) ** 2)

    W = 12 * S / (N**2 * (K**3 - K) - N * T_sum)
    return W, S


import pandas as pd

def build_table(rankings, R, final_ranks):
    K, N = rankings.shape

    data = {}

    for j in range(N):
        data[f"Փորձագետ {j+1}"] = rankings[:, j]

    data["Գումար R"] = R

    data["Վերջնական տեղ"] = final_ranks

    df = pd.DataFrame(data)
    df.index = [f"Գործոն {i+1}" for i in range(K)]

    df = df.sort_values("Վերջնական տեղ")

    return df

pairwise = input_pairwise()

scores = pairwise_to_scores(pairwise)

rankings = scores_to_rankings(scores)

R, final_ranks = compute_final_ranking(rankings)

print("\nՎերջնական դասակարգում")
for i, r in enumerate(final_ranks):
    print(f"Գործոն {i+1} → {r}")

matches = expert_matches(rankings, final_ranks)

T_sum = compute_Tj(matches)

W, S = kendall_W(rankings, R, T_sum)

print("\nKendall W =", round(W, 4))

Գործոնների քանակ K = 3
Փորձագետների քանակ N = 4

Մուտքագրիր միայն i < j զույգերը
Եթե i-ն գերադասելի է j-ից → 1, հակառակ դեպքում 0


Փորձագետ 1
1 vs 2: 1
1 vs 3: 1
2 vs 3: 1

Փորձագետ 2
1 vs 2: 1
1 vs 3: 1
2 vs 3: 0

Փորձագետ 3
1 vs 2: 0
1 vs 3: 0
2 vs 3: 0

Փորձագետ 4
1 vs 2: 0
1 vs 3: 0
2 vs 3: 1

Վերջնական դասակարգում
Գործոն 1 → 1
Գործոն 2 → 2
Գործոն 3 → 3
Փորձագետ 1: 3/3
Փորձագետ 2: 1/3
Փորձագետ 3: 1/3
Փորձագետ 4: 0/3
T1 = 24
T2 = 0
T3 = 0
T4 = 0

Kendall W = 0.0


In [ ]:
table = build_table(rankings, R, final_ranks)

print("\nԱղյուսակ (վերջնական արդյունքներով):\n")
print(table)


Աղյուսակ (վերջնական արդյունքներով):

          Փորձագետ 1  Փորձագետ 2  Փորձագետ 3  Փորձագետ 4  Գումար R  \
Գործոն 1         1.0         1.0         3.0         3.0       8.0   
Գործոն 2         2.0         3.0         2.0         1.0       8.0   
Գործոն 3         3.0         2.0         1.0         2.0       8.0   

          Վերջնական տեղ  
Գործոն 1              1  
Գործոն 2              2  
Գործոն 3              3  
